In [6]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import numpy as np
import sqlite3
from pathlib import Path

db_path = Path("Data/goodreads.db")

conn = sqlite3.connect(db_path)

print(f"Database created at: {db_path}")

Database created at: Data\goodreads.db


In [7]:
folder = Path("Data/Goodreads_Books/genres_top100")

keep_cols = [
    "name",
    "author",
    "genres",
    "pub_year",
    "star_rating",
    "num_ratings",
    "isbn_clean"
]

tables = []

for file in folder.glob("*.parquet"):
    print(f"Reading {file.name}")
    
    table = pq.read_table(file, columns=keep_cols)
    
    genre_col = pa.array([file.stem] * table.num_rows)
    table = table.append_column("source_genre", genre_col)
    
    tables.append(table)

combined_table = pa.concat_tables(
    tables,
    promote_options="default"
)

df = combined_table.to_pandas()

df["pub_year"] = pd.to_numeric(
    df["pub_year"],
    errors="coerce",
    downcast="integer"
)

df["source_genre"] = df["source_genre"].astype("category")

valid = df[
    (df["pub_year"] >= 1900) &
    (df["pub_year"] <= 2016)
].copy()

valid["genres_list"] = valid["genres"].apply(
    lambda g: [
        str(x).strip()
        for x in g
    ]
    if isinstance(g, (list, tuple, np.ndarray))
    else []
)

exploded = valid.explode("genres_list")

exploded = exploded[
    exploded["genres_list"].notna() &
    (exploded["genres_list"] != "")
]

exploded["genres_list"] = (
    exploded["genres_list"]
    .str.strip()
    .str.lower()
)

fiction_labels = {
    "fiction",
    "non-fiction",
    "nonfiction"
}

genre_only = exploded[
    ~exploded["genres_list"].isin(fiction_labels)
].copy()

print(f"valid: {valid.shape}")
print(f"genre_only: {genre_only.shape}")

Reading action.parquet
Reading adult.parquet
Reading adventure.parquet
Reading amazon.parquet
Reading american_history.parquet
Reading animals.parquet
Reading anthologies.parquet
Reading art.parquet
Reading audiobook.parquet
Reading bdsm.parquet
Reading biography.parquet
Reading biography_memoir.parquet
Reading book_club.parquet
Reading british_literature.parquet
Reading business.parquet
Reading chick_lit.parquet
Reading childrens.parquet
Reading christian.parquet
Reading christianity.parquet
Reading christian_fiction.parquet
Reading christmas.parquet
Reading classics.parquet
Reading comics.parquet
Reading comic_book.parquet
Reading contemporary.parquet
Reading contemporary_romance.parquet
Reading cookbooks.parquet
Reading cooking.parquet
Reading crime.parquet
Reading drama.parquet
Reading ebooks.parquet
Reading economics.parquet
Reading education.parquet
Reading erotica.parquet
Reading essays.parquet
Reading family.parquet
Reading fantasy.parquet
Reading fiction.parquet
Reading food.p

In [2]:
print(db_path.exists())

True


In [3]:
cursor = conn.cursor()

cursor.executescript("""
DROP TABLE IF EXISTS book_genres;
DROP TABLE IF EXISTS book_source_genres;
DROP TABLE IF EXISTS genres;
DROP TABLE IF EXISTS source_genres;
DROP TABLE IF EXISTS books;

CREATE TABLE books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    author TEXT,
    pub_year INTEGER,
    star_rating REAL,
    num_ratings INTEGER,
    isbn_clean TEXT UNIQUE
);

CREATE TABLE genres (
    genre_id INTEGER PRIMARY KEY AUTOINCREMENT,
    genre_name TEXT NOT NULL UNIQUE
);

CREATE TABLE source_genres (
    source_genre_id INTEGER PRIMARY KEY AUTOINCREMENT,
    source_genre TEXT NOT NULL UNIQUE
);

CREATE TABLE book_genres (
    book_id INTEGER NOT NULL,
    genre_id INTEGER NOT NULL,
    
    PRIMARY KEY (book_id, genre_id),
    
    FOREIGN KEY (book_id) REFERENCES books(book_id),
    FOREIGN KEY (genre_id) REFERENCES genres(genre_id)
);

CREATE TABLE book_source_genres (
    book_id INTEGER NOT NULL,
    source_genre_id INTEGER NOT NULL,
    
    PRIMARY KEY (book_id, source_genre_id),
    
    FOREIGN KEY (book_id) REFERENCES books(book_id),
    FOREIGN KEY (source_genre_id) REFERENCES source_genres(source_genre_id)
);
""")

conn.commit()

print("Tables created successfully.")

Tables created successfully.


In [10]:
books_df = valid[
    [
        "name",
        "author",
        "pub_year",
        "star_rating",
        "num_ratings",
        "isbn_clean"
    ]
].copy()

print(f"Initial book records: {len(books_df):,}")

Initial book records: 4,352,106


In [9]:
print(books_df.shape)
books_df.head()

(4352106, 6)


,name,author,pub_year,star_rating,num_ratings,isbn_clean
0,Sharpe's Devil,[Bernard Cornwell],1992,4.14,8141,9780060932299
1,Blood Fever,[Charlie Higson],2006,4.02,7516,9780786836628
2,Detektiv Conan vs. Kaito Kid,[Gosho Aoyama],2004,4.33,231,9783770476374
3,Marvel's Captain America: Sub Rosa,[David McDonald],2016,3.39,74,9781772752014
4,The Awakening,[Jerry Ahern],1984,3.87,305,9780821714782


In [11]:
print(
    f"Books with ISBN: "
    f"{books_df['isbn_clean'].notna().sum():,}"
)

print(
    f"Books without ISBN: "
    f"{books_df['isbn_clean'].isna().sum():,}"
)

print(
    f"Unique ISBNs: "
    f"{books_df['isbn_clean'].nunique():,}"
)

Books with ISBN: 3,521,260
Books without ISBN: 830,846
Unique ISBNs: 1,178,890


In [12]:
duplicate_isbn_count = (
    books_df["isbn_clean"]
    .duplicated(keep=False)
    .sum()
)

print(f"Records with duplicate ISBNs: {duplicate_isbn_count:,}")


Records with duplicate ISBNs: 3,891,881


In [13]:
books_df.head()

,name,author,pub_year,star_rating,num_ratings,isbn_clean
0,Sharpe's Devil,[Bernard Cornwell],1992,4.14,8141,9780060932299
1,Blood Fever,[Charlie Higson],2006,4.02,7516,9780786836628
2,Detektiv Conan vs. Kaito Kid,[Gosho Aoyama],2004,4.33,231,9783770476374
3,Marvel's Captain America: Sub Rosa,[David McDonald],2016,3.39,74,9781772752014
4,The Awakening,[Jerry Ahern],1984,3.87,305,9780821714782


In [14]:
books_df.info()


<class 'pandas.DataFrame'>
Index: 4352106 entries, 0 to 4820936
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   name         str    
 1   author       object 
 2   pub_year     int16  
 3   star_rating  float64
 4   num_ratings  int64  
 5   isbn_clean   str    
dtypes: float64(1), int16(1), int64(1), object(1), str(2)
memory usage: 384.8+ MB


In [15]:
valid[
    [
        "name",
        "author",
        "pub_year",
        "star_rating",
        "num_ratings",
        "isbn_clean",
        "source_genre"
    ]
].head(10)

,name,author,pub_year,star_rating,num_ratings,isbn_clean,source_genre
0,Sharpe's Devil,[Bernard Cornwell],1992,4.14,8141,9780060932299,action
1,Blood Fever,[Charlie Higson],2006,4.02,7516,9780786836628,action
2,Detektiv Conan vs. Kaito Kid,[Gosho Aoyama],2004,4.33,231,9783770476374,action
3,Marvel's Captain America: Sub Rosa,[David McDonald],2016,3.39,74,9781772752014,action
4,The Awakening,[Jerry Ahern],1984,3.87,305,9780821714782,action
5,Last of the Breed,[Louis L'Amour],1986,4.28,15161,NaN,action
6,Terrible Tuesday,[Don Pendleton],1979,4.02,292,9781497687622,action
7,Terminal Freeze,[Lincoln Child],2008,3.83,19403,9780385515511,action
8,"Eureka Seven: Psalms of Planets, Vol. 2",[Jinsei Kataoka],2005,4.05,268,9781594096914,action
9,Devil's Pass,[Sigmund Brouwer],2012,3.83,595,9781554699384,action


In [16]:
valid["source_genre"].value_counts().head(20)

source_genre
nonfiction            375997
fiction               316541
romance               200792
history               175469
fantasy               145872
childrens             117513
contemporary           95739
comics                 92396
mystery                90866
audiobook              78071
science_fiction        76069
young_adult            72466
historical_fiction     69820
biography              67625
poetry                 64198
picture_books          62899
historical             62185
short_stories          60767
graphic_novels         58917
paranormal             56824
Name: count, dtype: int64

In [17]:
books_df["author"] = books_df["author"].apply(
    lambda x: ", ".join(x) if isinstance(x, list) else str(x)
)

In [18]:
books_df["author"].head()

0      ['Bernard Cornwell']
1        ['Charlie Higson']
2          ['Gosho Aoyama']
3    ['David     McDonald']
4           ['Jerry Ahern']
Name: author, dtype: str

In [19]:
import ast
import pandas as pd

def clean_author(value):
    if pd.isna(value):
        return None
    
    try:
        authors = ast.literal_eval(value)
        
        if isinstance(authors, list):
            return ", ".join(
                str(author).strip()
                for author in authors
            )
        
        return str(authors).strip()
    
    except (ValueError, SyntaxError):
        return str(value).strip()

books_df["author"] = books_df["author"].apply(clean_author)

In [20]:
books_df["author"].head(10)

0      Bernard Cornwell
1        Charlie Higson
2          Gosho Aoyama
3    David     McDonald
4           Jerry Ahern
5         Louis L'Amour
6         Don Pendleton
7         Lincoln Child
8        Jinsei Kataoka
9       Sigmund Brouwer
Name: author, dtype: str

In [21]:
books_df["author"] = (
    books_df["author"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [22]:
books_df["author"].head(10)

0    Bernard Cornwell
1      Charlie Higson
2        Gosho Aoyama
3      David McDonald
4         Jerry Ahern
5       Louis L'Amour
6       Don Pendleton
7       Lincoln Child
8      Jinsei Kataoka
9     Sigmund Brouwer
Name: author, dtype: str

In [23]:
print("Total records:", f"{len(books_df):,}")
print("ISBN present:", f"{books_df['isbn_clean'].notna().sum():,}")
print("ISBN missing:", f"{books_df['isbn_clean'].isna().sum():,}")
print("Unique ISBNs:", f"{books_df['isbn_clean'].nunique():,}")

duplicate_isbn = (
    books_df.loc[
        books_df["isbn_clean"].notna(),
        "isbn_clean"
    ]
    .duplicated()
    .sum()
)

print("Duplicate ISBN records:", f"{duplicate_isbn:,}")

Total records: 4,352,106
ISBN present: 3,521,260
ISBN missing: 830,846
Unique ISBNs: 1,178,890
Duplicate ISBN records: 2,342,370


In [24]:
books_work = valid[
    [
        "name",
        "author",
        "pub_year",
        "star_rating",
        "num_ratings",
        "isbn_clean"
    ]
].copy()

In [25]:
import ast
import pandas as pd

def clean_author(value):
    if pd.isna(value):
        return None
    
    try:
        authors = ast.literal_eval(value)
        
        if isinstance(authors, list):
            return ", ".join(
                str(author).strip()
                for author in authors
            )
        
        return str(authors).strip()
    
    except (ValueError, SyntaxError):
        return str(value).strip()

books_work["author"] = books_work["author"].apply(clean_author)

books_work["author"] = (
    books_work["author"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [26]:
books_work[["name", "author"]].head(10)

,name,author
0,Sharpe's Devil,['Bernard Cornwell']
1,Blood Fever,['Charlie Higson']
2,Detektiv Conan vs. Kaito Kid,['Gosho Aoyama']
3,Marvel's Captain America: Sub Rosa,['David McDonald']
4,The Awakening,['Jerry Ahern']
5,Last of the Breed,"[""Louis L'Amour""]"
6,Terrible Tuesday,['Don Pendleton']
7,Terminal Freeze,['Lincoln Child']
8,"Eureka Seven: Psalms of Planets, Vol. 2",['Jinsei Kataoka']
9,Devil's Pass,['Sigmund Brouwer']


In [27]:
books_work["name_clean"] = (
    books_work["name"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

books_work["author_clean"] = (
    books_work["author"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [28]:
books_work["book_key"] = np.where(
    books_work["isbn_clean"].notna(),
    "ISBN:" + books_work["isbn_clean"].astype(str),
    "TITLE:" +
    books_work["name_clean"] + "|" +
    books_work["author_clean"] + "|" +
    books_work["pub_year"].astype(str)
)

In [29]:
print("Total records:", f"{len(books_work):,}")
print("Unique book keys:", f"{books_work['book_key'].nunique():,}")

Total records: 4,352,106
Unique book keys: 1,487,804


In [30]:
books_df = (
    books_work
    .sort_values(
        ["book_key", "num_ratings"],
        ascending=[True, False]
    )
    .drop_duplicates(
        subset="book_key",
        keep="first"
    )
    .copy()
)

In [31]:
books_df = books_df.reset_index(drop=True)

books_df["book_id"] = books_df.index + 1

In [32]:
books_df[
    [
        "book_id",
        "name",
        "author",
        "pub_year",
        "star_rating",
        "num_ratings",
        "isbn_clean"
    ]
].head(10)

,book_id,name,author,pub_year,star_rating,num_ratings,isbn_clean
0,1,Calabar - O Elogio da Traição,['Chico Buarque'],1966,3.66,172,0000000709996
1,2,Hetkiä historiassa: Cultural History - Kulttuu...,['Henri Terho'],2002,3.00,6,0000014581949
2,3,I S A: Hidup dan Ajaran Sang Masiha,['Anand Krishna'],2008,4.37,52,0000020499390
3,4,Learning and Instruction: Theory into Practice,['Margaret E. Gredler'],1991,3.54,41,0000131591231
4,5,Free Speech: A Very Short Introduction,['Nigel Warburton'],2009,3.64,870,0000199232350
5,6,Motion Graphic Design: Applied History and Aes...,['Jon Krasner'],2008,4.04,27,0000240809890
6,7,"White Walls, Designer Dresses: The Fashioning ...",['Mark Wigley'],1996,4.04,27,0000262731452
7,8,دفاتر التدوين: الدفتر الأول: خلسات الكرى,['Gamal al-Ghitani'],2003,3.11,168,00003543/2003
8,9,دفاتر التدوين : الدفتر الثاني : دنى فتدلى,['Gamal al-Ghitani'],1998,3.48,86,00003544/2003
9,10,دفاتر التدوين : الدفتر الثالث : رشحات الحمراء,['Gamal al-Ghitani'],2003,3.53,77,00003545/2003


In [33]:
print("Total records:", f"{len(books_work):,}")
print("Unique book keys:", f"{books_work['book_key'].nunique():,}")
print("Unique books:", f"{len(books_df):,}")

Total records: 4,352,106
Unique book keys: 1,487,804
Unique books: 1,487,804


In [34]:
import sqlite3
from pathlib import Path

db_path = Path("Data/goodreads.db")

# Remove the old database if you created one during testing
if db_path.exists():
    db_path.unlink()

conn = sqlite3.connect(db_path)

# Enforce foreign-key relationships
conn.execute("PRAGMA foreign_keys = ON;")

print(f"Database created: {db_path}")

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'Data\\goodreads.db'

In [35]:
conn.close()

In [36]:
print("Database connection closed.")

Database connection closed.


In [37]:
import sqlite3
from pathlib import Path

db_path = Path("Data/goodreads.db")

if db_path.exists():
    db_path.unlink()

conn = sqlite3.connect(db_path)

conn.execute("PRAGMA foreign_keys = ON;")

print(f"Database created: {db_path}")

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'Data\\goodreads.db'

In [38]:
import sqlite3
from pathlib import Path

db_path = Path("Data/goodreads_capstone.db")

conn = sqlite3.connect(db_path)

conn.execute("PRAGMA foreign_keys = ON;")

print(f"Database created: {db_path}")

Database created: Data\goodreads_capstone.db


In [39]:
print(conn)

In [40]:
cursor = conn.cursor()

cursor.executescript("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    author TEXT,
    pub_year INTEGER,
    star_rating REAL,
    num_ratings INTEGER,
    isbn_clean TEXT,
    book_key TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS genres (
    genre_id INTEGER PRIMARY KEY AUTOINCREMENT,
    genre_name TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS source_genres (
    source_genre_id INTEGER PRIMARY KEY AUTOINCREMENT,
    source_genre TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS book_genres (
    book_id INTEGER NOT NULL,
    genre_id INTEGER NOT NULL,

    PRIMARY KEY (book_id, genre_id),

    FOREIGN KEY (book_id)
        REFERENCES books(book_id),

    FOREIGN KEY (genre_id)
        REFERENCES genres(genre_id)
);

CREATE TABLE IF NOT EXISTS book_source_genres (
    book_id INTEGER NOT NULL,
    source_genre_id INTEGER NOT NULL,

    PRIMARY KEY (book_id, source_genre_id),

    FOREIGN KEY (book_id)
        REFERENCES books(book_id),

    FOREIGN KEY (source_genre_id)
        REFERENCES source_genres(source_genre_id)
);
""")

conn.commit()

print("All tables created successfully.")

All tables created successfully.


In [41]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn
)

tables

,name
0,book_genres
1,book_source_genres
2,books
3,genres
4,source_genres
5,sqlite_sequence


In [42]:
books_sql = books_df[
    [
        "book_id",
        "name",
        "author",
        "pub_year",
        "star_rating",
        "num_ratings",
        "isbn_clean",
        "book_key"
    ]
].copy()

print(f"Books ready for SQL: {len(books_sql):,}")

Books ready for SQL: 1,487,804


In [43]:
books_sql.to_sql(
    "books",
    conn,
    if_exists="append",
    index=False,
    chunksize=10000
)

print(f"Inserted {len(books_sql):,} books.")

Inserted 1,487,804 books.


In [44]:
book_count = pd.read_sql_query(
    """
    SELECT COUNT(*) AS book_count
    FROM books;
    """,
    conn
)

print(book_count)

   book_count
0     1487804


In [45]:
pd.read_sql_query(
    """
    SELECT *
    FROM books
    LIMIT 10;
    """,
    conn
)

,book_id,name,author,pub_year,star_rating,num_ratings,isbn_clean,book_key
0,1,Calabar - O Elogio da Traição,['Chico Buarque'],1966,3.66,172,0000000709996,ISBN:0000000709996
1,2,Hetkiä historiassa: Cultural History - Kulttuu...,['Henri Terho'],2002,3.00,6,0000014581949,ISBN:0000014581949
2,3,I S A: Hidup dan Ajaran Sang Masiha,['Anand Krishna'],2008,4.37,52,0000020499390,ISBN:0000020499390
3,4,Learning and Instruction: Theory into Practice,['Margaret E. Gredler'],1991,3.54,41,0000131591231,ISBN:0000131591231
4,5,Free Speech: A Very Short Introduction,['Nigel Warburton'],2009,3.64,870,0000199232350,ISBN:0000199232350
5,6,Motion Graphic Design: Applied History and Aes...,['Jon Krasner'],2008,4.04,27,0000240809890,ISBN:0000240809890
6,7,"White Walls, Designer Dresses: The Fashioning ...",['Mark Wigley'],1996,4.04,27,0000262731452,ISBN:0000262731452
7,8,دفاتر التدوين: الدفتر الأول: خلسات الكرى,['Gamal al-Ghitani'],2003,3.11,168,00003543/2003,ISBN:00003543/2003
8,9,دفاتر التدوين : الدفتر الثاني : دنى فتدلى,['Gamal al-Ghitani'],1998,3.48,86,00003544/2003,ISBN:00003544/2003
9,10,دفاتر التدوين : الدفتر الثالث : رشحات الحمراء,['Gamal al-Ghitani'],2003,3.53,77,00003545/2003,ISBN:00003545/2003


In [46]:
genres_df = pd.DataFrame({
    "genre_name": (
        genre_only["genres_list"]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
        .unique()
    )
})

genres_df = genres_df[
    genres_df["genre_name"] != ""
].copy()

genres_df = genres_df.sort_values(
    "genre_name"
).reset_index(drop=True)

print(f"Unique Goodreads genres: {len(genres_df):,}")

Unique Goodreads genres: 1,433


In [47]:
genres_df.to_sql(
    "genres",
    conn,
    if_exists="append",
    index=False
)

print("Genres inserted.")

Genres inserted.


In [48]:
pd.read_sql_query(
    """
    SELECT COUNT(*) AS genre_count
    FROM genres;
    """,
    conn
)

,genre_count
0,1433


In [49]:
pd.read_sql_query(
    """
    SELECT *
    FROM genres
    ORDER BY genre_name
    LIMIT 25;
    """,
    conn
)

,genre_id,genre_name
0,1,10th century
1,2,11th century
2,3,12th century
3,4,13th century
4,5,14th century
5,6,15th century
6,7,16th century
7,8,17th century
8,9,1864 shenandoah campaign
9,10,18th century


In [50]:
source_genres_df = pd.DataFrame({
    "source_genre": (
        valid["source_genre"]
        .astype(str)
        .str.strip()
        .str.lower()
        .unique()
    )
})

source_genres_df = source_genres_df.sort_values(
    "source_genre"
).reset_index(drop=True)

print(f"Source genres: {len(source_genres_df):,}")

Source genres: 100


In [51]:
source_genres_df.to_sql(
    "source_genres",
    conn,
    if_exists="append",
    index=False
)

print("Source genres inserted.")

Source genres inserted.


In [52]:
pd.read_sql_query(
    """
    SELECT *
    FROM source_genres
    ORDER BY source_genre;
    """,
    conn
)

,source_genre_id,source_genre
0,1,action
1,2,adult
2,3,adventure
3,4,amazon
4,5,american_history
...,...,...
95,96,urban_fantasy
96,97,vampires
97,98,war
98,99,westerns


In [53]:
books_lookup = pd.read_sql_query(
    """
    SELECT book_id, book_key
    FROM books;
    """,
    conn
)

genres_lookup = pd.read_sql_query(
    """
    SELECT genre_id, genre_name
    FROM genres;
    """,
    conn
)

source_lookup = pd.read_sql_query(
    """
    SELECT source_genre_id, source_genre
    FROM source_genres;
    """,
    conn
)

print(f"Books: {len(books_lookup):,}")
print(f"Genres: {len(genres_lookup):,}")
print(f"Source genres: {len(source_lookup):,}")

Books: 1,487,804
Genres: 1,433
Source genres: 100


In [54]:
genre_relationships = genre_only[
    [
        "name",
        "author",
        "pub_year",
        "isbn_clean",
        "genres_list"
    ]
].copy()

print(f"Starting genre relationships: {len(genre_relationships):,}")

Starting genre relationships: 24,100,598


In [55]:
genre_relationships["author"] = (
    genre_relationships["author"]
    .apply(clean_author)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [56]:
genre_relationships["name_clean"] = (
    genre_relationships["name"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

genre_relationships["author_clean"] = (
    genre_relationships["author"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [57]:
genre_relationships["book_key"] = np.where(
    genre_relationships["isbn_clean"].notna(),
    "ISBN:" + genre_relationships["isbn_clean"].astype(str),
    "TITLE:" +
    genre_relationships["name_clean"] + "|" +
    genre_relationships["author_clean"] + "|" +
    genre_relationships["pub_year"].astype(str)
)

In [58]:
genre_relationships = genre_relationships.merge(
    books_lookup,
    on="book_key",
    how="inner"
)

print(
    f"Matched genre records: "
    f"{len(genre_relationships):,}"
)

Matched genre records: 24,100,598


In [59]:
genre_relationships = genre_relationships.merge(
    genres_lookup,
    left_on="genres_list",
    right_on="genre_name",
    how="inner"
)

In [60]:
book_genres_sql = genre_relationships[
    [
        "book_id",
        "genre_id"
    ]
].drop_duplicates()

print(
    f"Unique book/genre relationships: "
    f"{len(book_genres_sql):,}"
)

Unique book/genre relationships: 4,981,665


In [61]:
book_genres_sql.to_sql(
    "book_genres",
    conn,
    if_exists="append",
    index=False,
    chunksize=10000
)

print("Book/genre relationships inserted.")

Book/genre relationships inserted.


In [62]:
source_relationships = valid[
    [
        "name",
        "author",
        "pub_year",
        "isbn_clean",
        "source_genre"
    ]
].copy()

In [63]:
source_relationships["author"] = (
    source_relationships["author"]
    .apply(clean_author)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [64]:
source_relationships["name_clean"] = (
    source_relationships["name"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

source_relationships["author_clean"] = (
    source_relationships["author"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [65]:
source_relationships["book_key"] = np.where(
    source_relationships["isbn_clean"].notna(),
    "ISBN:" + source_relationships["isbn_clean"].astype(str),
    "TITLE:" +
    source_relationships["name_clean"] + "|" +
    source_relationships["author_clean"] + "|" +
    source_relationships["pub_year"].astype(str)
)

In [66]:
source_relationships = source_relationships.merge(
    books_lookup,
    on="book_key",
    how="inner"
)

source_relationships = source_relationships.merge(
    source_lookup,
    on="source_genre",
    how="inner"
)

In [67]:
book_source_sql = source_relationships[
    [
        "book_id",
        "source_genre_id"
    ]
].drop_duplicates()

print(
    f"Unique book/source relationships: "
    f"{len(book_source_sql):,}"
)

Unique book/source relationships: 4,291,569


In [68]:
book_source_sql.to_sql(
    "book_source_genres",
    conn,
    if_exists="append",
    index=False,
    chunksize=10000
)

print("Book/source-genre relationships inserted.")

Book/source-genre relationships inserted.


In [69]:
cursor.executescript("""
CREATE INDEX IF NOT EXISTS idx_books_pub_year
    ON books(pub_year);

CREATE INDEX IF NOT EXISTS idx_books_rating
    ON books(star_rating);

CREATE INDEX IF NOT EXISTS idx_books_num_ratings
    ON books(num_ratings);

CREATE INDEX IF NOT EXISTS idx_books_isbn
    ON books(isbn_clean);

CREATE INDEX IF NOT EXISTS idx_book_genres_book
    ON book_genres(book_id);

CREATE INDEX IF NOT EXISTS idx_book_genres_genre
    ON book_genres(genre_id);

CREATE INDEX IF NOT EXISTS idx_book_source_book
    ON book_source_genres(book_id);

CREATE INDEX IF NOT EXISTS idx_book_source_genre
    ON book_source_genres(source_genre_id);
""")

conn.commit()

print("Indexes created.")

Indexes created.


In [70]:
for table in [
    "books",
    "genres",
    "source_genres",
    "book_genres",
    "book_source_genres"
]:
    
    count = pd.read_sql_query(
        f"SELECT COUNT(*) AS count FROM {table};",
        conn
    ).iloc[0, 0]
    
    print(f"{table:25} {count:,}")

books                     1,487,804
genres                    1,433
source_genres             100
book_genres               4,981,665
book_source_genres        4,291,569
